# Final Exam: Phase 2: CIFAR-10 with Adam (lr=0.001, lr=0.0003)
Mỗi notebook chứa sẵn mã nguồn và test đã review; không clone nhánh GitHub đang thay đổi.
Chọn GPU, bật Internet để tải CIFAR và bảo đảm môi trường có torch, torchvision, numpy, pandas, pillow, pytest.
Cấu hình cố định: 200 epochs, seed 42, train/validation 45.000/5.000; chọn checkpoint bằng validation.
Test chính thức được đánh giá sau khi hoàn tất mỗi thực nghiệm; không dùng test để chọn learning rate.

Nếu cần chạy nhiều phiên, đặt EPOCHS_PER_SESSION (ví dụ 50); tải recovery.zip về trước khi phiên kết thúc.
Phiên tiếp theo: giải nén recovery.zip thành Kaggle Dataset, attach dataset, rồi đặt RESUME_ROOT trỏ đến thư mục chứa các run.
Resume yêu cầu cùng phiên bản torch/torchvision/NumPy/Pillow, loại GPU, mã nguồn và cấu hình.
Kết quả chỉ được gắn nhãn results khi cả hai run đủ 200 epochs và toàn bộ artifact được xác thực.


In [ ]:
import base64, hashlib, io, json, os, sys, zipfile
from pathlib import Path
import subprocess, importlib.util

BUNDLE_SHA256 = '7c4b3888eeec557b153339f48f244e47a4b8e981739260994dd70088c941c2bf'
SOURCE_BUNDLE = '...'
raw = base64.b64decode(SOURCE_BUNDLE)
assert hashlib.sha256(raw).hexdigest() == BUNDLE_SHA256
PROJECT = Path('/kaggle/working') / ('TickNets-' + BUNDLE_SHA256[:16])
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    manifest = json.loads(archive.read('source_manifest.json'))
    for name, expected in manifest.items():
        payload = archive.read(name)
        assert hashlib.sha256(payload).hexdigest() == expected
        target = (PROJECT / name).resolve()
        assert target.is_relative_to(PROJECT.resolve())
        if target.exists() and target.read_bytes() != payload:
            raise RuntimeError('Extracted source was modified: ' + name)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
    (PROJECT / 'source_manifest.json').write_bytes(archive.read('source_manifest.json'))
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
# Install only missing test/report dependencies; preserve Kaggle's torch/CUDA build.
requirements = {'pytest': 'pytest>=8,<10', 'pandas': 'pandas', 'numpy': 'numpy', 'PIL': 'pillow'}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch, torchvision
from torch.utils.flop_counter import FlopCounterMode
assert torch.cuda.is_available(), 'Select a Kaggle GPU accelerator before running.'
print('Source bundle:', BUNDLE_SHA256)
print('Runtime:', torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Validate the embedded code before spending GPU time.
environment = dict(os.environ, CUDA_VISIBLE_DEVICES='-1', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
subprocess.run([sys.executable, '-m', 'pytest', *['tests/test_train_cifar.py', 'tests/test_cifar_data.py', 'tests/test_download_cifar.py', 'tests/test_cifar_recovery.py', 'tests/test_cifar_contracts.py', 'tests/test_cifar_download_integrity.py', 'tests/test_cifar_results.py'],
                '-q', '-p', 'no:cacheprovider'], cwd=PROJECT, env=environment, check=True)


In [ ]:
# None: run all remaining epochs. Integer: epochs per experiment for this session.
EPOCHS_PER_SESSION = None
# Path to the attached recovery dataset containing the experiment folders (not the zip itself).
RESUME_ROOT = None


In [ ]:
command = [sys.executable, 'scripts/run_kaggle_phase.py', '--phase', '2']
if EPOCHS_PER_SESSION is not None:
    command += ['--epochs-per-session', str(EPOCHS_PER_SESSION)]
if RESUME_ROOT is not None:
    command += ['--resume-root', str(RESUME_ROOT)]
subprocess.run(command, cwd=PROJECT, check=True)
